# Background Information - What is WiFi and how does it work?

Every radio signal is fundamentally an electromagnetic wave travelling through the air oscillating at some frequency. Different technologies get assigned different frequency ranges (mostly by international agreement) and WiFi signals are mainly around 2.4GHz. WiFi is a protocol that allows devices to wirelessly exchange data with each other. The protocol in short:
* A subcarrier is a radio wave at one specific frequency. 
* Your data starts as bits inside your phone. These bits then get grouped into small chunks (allows better information encoding) and then split across subcarriers.
* Each subcarrier gets its own ongoing stream of these chunks which are then used to transform the subcarriers amplitude and phase (based on a lookup table).
* These subcarriers then travel through the air or through walls, picking up some real-world distortion.
* The router then measures each subcarrier's actual amplitude/phase (its CSI) and decodes the information turning everything back into the original data.
* This whole technique is called Orthogonal Frequency-Divison Multplexing (OFDM). The transmission of a group of data is also called a **packet**.

Channel State Information (CSI) is data that WiFi hardware (like a router) computes as a part of its normal operations. It describes how the radio signal was distorted travelling from a transmitter to a receiver (i.e., did the signal get weaker or did its phase shift). The frequency range in which the subcarriers operate in is called a channel. So around 2.4GHz there are multiple channels which usually have a range of 20MHz. The main benefit for having multiple channels is so that your network has its own frequency space. If your neighbor is using channel 6, your WiFi would use channel 11 so that there is no interference.

## The Aim of the Report

With an understanding of how WiFi works, the aim of this report can be better understood. Research shows that when a person walks through a room their body can absorb/reflect WiFi signals which can affect CSI in a detectable way. The goal of this report is to predict human presence based on these distortions.

### Wi-Fi CSI presence detection - Stage 1 data analysis

Room4 of the IEEE DataPort *CSI Dataset for WiFi Based Human Detection* (DOI 10.21227/smn7-jb67):
empty room (label 0) vs one person present (label 1).
Each magnitude CSV = one ~70 s recording; rows = packets (time), 49 columns = subcarriers
(47 used after dropping the two band edges); no header, no timestamps.

## Setup and data location
- **JupyterHub:** upload `data_room4.zip` (plus the optional `data_room4_phase.zip` and `data_lab_static.zip`) next to this notebook; the cell below unzips them once.
- **Local repo:** uses the project's `CSV Files/` folder one level up.

In [4]:
import glob
import os
import zipfile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import os


# look for the dataset in the current working directory: 
for zip_path in sorted(glob.glob("data_*.zip")): #looks in current folder for zip files with names like data_room4.zip
    with zipfile.ZipFile(zip_path) as z:
        if not os.path.exists(z.namelist()[-1]):     # check if that file has been extracted
            z.extractall(".")  #unzip any data_*.zip in the same folder where .ipynb file lives (JupyterHub)
DATA = "CSV Files" if os.path.isdir("CSV Files") else "../CSV Files" #does a folder name CSV file exist, if yes name it CSV Files
assert os.path.isdir(DATA), "CSV Files/ not found - upload data_room4.zip next to this notebook"


# make the paths easy names
FOLDERS = {
    "empty":      f"{DATA}/room4_iqbal/Static_noperson_channel_11_20mhz/csi_magnitude_data",
    "person":     f"{DATA}/room4_iqbal/Dynamic_1_person_ch11_20mhz/csi_magnitude_data",
    # not used for training: only checked to decide what "Static" means
    "lab_static": f"{DATA}/LAB_Final Year Project/Static_Lab_ch6_20mhz/csi_magnitude_data",
}
# complex (amplitude + phase) files: only read in the phase check below, never used as features
COMPLEX_FOLDERS = {name: FOLDERS[name].replace("csi_magnitude_data", "csi_complex_data")
                   for name in ("empty", "person")} #compact way to make the path names same as above with csi_comple_data instead of csi_magnitude_data
N_PHASE_FILES = 10   # first 10 recordings per class are enough to show the pattern
EXPECTED_FILES = {"empty": 10, "person": 85, "lab_static": 91} #how many files should be in each folder
DROP_SUBCARRIERS = [0, 48]   # band edges (noisy, so drop), see "Subcarrier selection" below
FILE_S = 70    # seconds between consecutive file start times (from the filenames)
WIN_S = 2      # window length in seconds
FIG_DIR = "figures"
os.makedirs(FIG_DIR, exist_ok=True)

## Helpers
This section loads the data and defines helpers for preprocessing.
1. First find & load the CSVs (list_csv, load_raw, load, load_complex)
2. Summarize signal behaviour per file (iqr, spread, summarise_folder
3. Chop recordings into fixed length windows for individual training examples
4. Organize metadata (timestamps and split data by files)

In [5]:
def list_csv(folder): #given a folder find every.csv file and return the file paths as a sorted list
    return sorted(glob.glob(folder + "/*.csv"))


def load_raw(path): #makes a dataframe out of all the files
    return pd.read_csv(path, header=None)    # no header row: the first line is data; all 49 subcarriers


def load(path): #preprocessing - drop subcarriers 0 and 48 to reduce noise
    return load_raw(path).drop(columns=DROP_SUBCARRIERS)  


def load_complex(path): 
    """Complex CSI from a csi_complex_data file; values are written like '284+410i'.
    thus it requires to convert each string to an actual python complex number, but it requires j instead of i"""
    text = pd.read_csv(path, header=None, dtype=str)
    return text.apply(lambda col: col.str.replace("i", "j").map(complex)).to_numpy()


def iqr(df): #describes the range the middle 50% of datapoints span
    return df.quantile(0.75) - df.quantile(0.25)   # one number per subcarrier


def spread(df):
    """Takes the median of the 47 IQR values - "how variable is this files signal" """
    return iqr(df).median()


def window(df, start_s, length_s=WIN_S, file_s=FILE_S):
    """Cut a length_s-second window starting at start_s, using this file's own packet rate."""
    rate = len(df) / file_s              # estimated packets per second for this file, ex: 500 packets / 70 seconds
    a = int(start_s * rate)              #row index corresponds to 10 sec mark
    b = a + int(length_s * rate)         #row index corresponds to 12 sec mark
    return df.iloc[a:b].to_numpy()           # shape: (packets, 47)


def summarise_folder(files):
    """Load each file ONCE and return its packet count and spread."""
    packets, spreads = [], []
    for f in files:
        df = load(f)
        packets.append(len(df))
        spreads.append(spread(df))
    return packets, spreads


def start_time(path):
    """Recording start time from a filename like csi_20241204_020632_magnitude.csv."""
    stamp = os.path.basename(path).split("_")[1:3]            # ['20241204', '020632']
    return pd.to_datetime("_".join(stamp), format="%Y%m%d_%H%M%S")

#split data into train/validation/test. Done by file now by row - avoids data leakage. so many files go into each 
def split_files(files, n_train, n_val):
    return {"train": files[:n_train],
            "val":   files[n_train:n_train + n_val],
            "test":  files[n_train + n_val:]}

## Inspect one file per class
Data validation before computing anything.
1. Find all the CSVs for each category
2. Confirm all files have been unzipped correctly
3. Load one representative file from "empty" and "person"
4. Visually confirm: correct shpae, no missing values, and some early on stats

In [6]:
#list_csv is being called on each folder and returning the sorted list of csv files found in it
files = {name: list_csv(folder) for name, folder in FOLDERS.items()} #.items gives you each key value pair, for name, folder unpacks  those values into variables name and folder
print({name: len(f) for name, f in files.items()}) #num of files in each

# Makes sure you have all the files: a half-finished unzip leaves too few files; if this fails, delete "CSV Files/" and re-run from the top
for name, n in EXPECTED_FILES.items():
    assert len(files[name]) in (n, 0 if name == "lab_static" else n), (name, len(files[name]))
#first of file in each folder - this is cleaned data
first = {name: load(files[name][0]) for name in ("empty", "person")}

#Print diagnostic for each sample file
for name, df in first.items():
    print(f"\n=== {name}: {len(files[name])} files, first = {os.path.basename(files[name][0])}") #header: folder name, total files, and file being inspected
    print("shape (packets, subcarriers):", df.shape)
    print("missing values:", df.isna().sum().sum())
    display(pd.DataFrame({"std": df.std(), "IQR": iqr(df)}).T.round(1))

{'empty': 0, 'person': 0, 'lab_static': 0}


AssertionError: ('empty', 0)

## Subcarrier comparison - Person vs Empty
An exploratory plot used to compare the wave form of single subcarrier from a room with a person vs. a room without a person.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharey=True, constrained_layout=True)
for ax, name, title in zip(axes, ("empty", "person"),
                           ("room4, empty (Static_noperson)", "room4, 1 person moving (Dynamic_1_person)")):
    ax.plot(first[name][1])
    ax.set_title(f"{title} - subcarrier 1")
    ax.set_xlabel("packet index")
    ax.set_ylabel("amplitude")
plt.savefig(f"{FIG_DIR}/fig_raw_subcarrier1.png", dpi=150)
plt.show()

## Justifying dropping the edge-subcarriers
Checks across every file in "empty" and "person" folders whether subcarriers 0 and 48 are consistently noisier (higher IQR) than a typical subcarrier. edge_table is a summary table justifying the decision

In [ ]:
#measuring across every file is subcarrier 0 or 48 consistently more variable than a typical subcarrier in the same file
ratio_rows = []
for name in ("empty", "person"):
    ratios = []
    for f in files[name]:
        r = iqr(load_raw(f))
        inner = r.drop(index=DROP_SUBCARRIERS).median()
        ratios.append((r[0] / inner, r[48] / inner))
    ratios = np.array(ratios)
    for i, sc in enumerate(DROP_SUBCARRIERS):
        ratio_rows.append({"class": name, "subcarrier": sc, "files": len(ratios),
                           "files with ratio > 1": int((ratios[:, i] > 1).sum()),
                           "median ratio": np.median(ratios[:, i]),
                           "min": ratios[:, i].min(), "max": ratios[:, i].max()})
edge_table = pd.DataFrame(ratio_rows).round(2)
edge_table.to_csv(f"{FIG_DIR}/table_edge_subcarriers.csv", index=False)
edge_table

## Feature selection: why phase is not used
This checks whether CSI phase carries usable information or is it just hardware noise. Raw CSI phase from commodity Wi-Fi cards are known to carry random per-packet offsets. For the first 10 recordings of each class ("empty" and "person") using subcarriers 1-47, we compute three diagnostics per file and summarize them into one row per class.
1. Circular variance of the phase over time (0 = constant phase, 1 = uniformly random)
2. the median phase jump between consecutive packets (uniformly random phase gives pi/2 = 1.57 rad)
3. And for comparison, the relative spread of the magnitude (IQR / median).

In [ ]:
phase_rows = []
for name, folder in COMPLEX_FOLDERS.items():
    complex_files = list_csv(folder)[:N_PHASE_FILES]
    if not complex_files:
        print(f"{name}: no complex files found - upload data_room4_phase.zip to run this check")
        continue
    circ_var, jump, mag_rel = [], [], []
    for f in complex_files:
        c = load_complex(f)[:, 1:48]                    # inner subcarriers, same as the features
        phase, mag = np.angle(c), np.abs(c)
        circ_var.append(np.median(1 - np.abs(np.mean(np.exp(1j * phase), axis=0))))
        jump.append(np.median(np.abs(np.angle(np.exp(1j * np.diff(phase, axis=0))))))
        q75, q25 = np.percentile(mag, [75, 25], axis=0)
        mag_rel.append(np.median((q75 - q25) / np.median(mag, axis=0)))
    phase_rows.append({"class": name, "files": len(complex_files),
                       "phase circular variance": np.median(circ_var),
                       "median phase jump (rad)": np.median(jump),
                       "magnitude IQR / median": np.median(mag_rel)})
phase_table = pd.DataFrame(phase_rows).round(3)
phase_table.to_csv(f"{FIG_DIR}/table_phase_check.csv", index=False)
phase_table

## Full-dataset spread summary and 'lab_static' label check
Here every file is loaded and as well as compute the spread. Includes a summary for each class. The code also investigates if 'lab_static' can be trusted

In [ ]:
summary = {name: summarise_folder(files[name]) for name in FOLDERS if files[name]}

for name, (packets, spreads) in summary.items():
    print(f"{name:10s} files={len(spreads):3d}  median IQR min={min(spreads):6.1f}  max={max(spreads):6.1f}")

# label check: if the Lab "Static" folder was loaded, compare its spread with room4
if "lab_static" in summary:
    lab = summary["lab_static"][1]
    print(f"\nLab 'Static' range {min(lab):.1f}-{max(lab):.1f} vs room4 empty max {max(summary['empty'][1]):.1f}, "
          f"person min {min(summary['person'][1]):.1f} -> inside the person range, so 'Static' is ambiguous "
          f"(empty in a noisier room, or a still person) and the Lab folders are excluded")

## Dataset statistics for room 4
This summary table combines: file counts, packet counts, duration, how many training samples after windowing, and repeat of earlier spreads for comparing "empty" vs "person" classes.

In [ ]:
rows = []
for name, label in (("empty", 0), ("person", 1)):
    packets, spreads = summary[name]
    rows.append({
        "class": name,
        "label": label,
        "files": len(packets),
        "packets (total)": sum(packets),
        "duration (s, approx.)": len(packets) * FILE_S,
        "packets/s (median)": round(np.median(packets) / FILE_S),
        f"{WIN_S}-s windows (non-overlapping)": len(packets) * (FILE_S // WIN_S),
        "median IQR (min-max)": f"{min(spreads):.1f}-{max(spreads):.1f}",
    })
table = pd.DataFrame(rows)
table.to_csv(f"{FIG_DIR}/table_dataset_stats.csv", index=False)
table

## Train / validation / test split and data leakage check
Splits each class's files into train/val/test sets using fixed counts. The splitting is done by whole files so a single recording never gets divided across sets.

In [ ]:
splits = {
    "empty":  split_files(files["empty"],  n_train=6,  n_val=2),    # 6 / 2 / 2
    "person": split_files(files["person"], n_train=51, n_val=17),   # 51 / 17 / 17
}

split_rows = []
for name, parts in splits.items():
    for part, part_files in parts.items():
        split_rows.append({
            "class": name,
            "set": part,
            "files": len(part_files),
            f"{WIN_S}-s windows": len(part_files) * (FILE_S // WIN_S),
            "first start": start_time(part_files[0]),
            "last start": start_time(part_files[-1]),
        })
split_table = pd.DataFrame(split_rows)
split_table.to_csv(f"{FIG_DIR}/table_split.csv", index=False)

# no file may appear in two sets
all_split = [f for parts in splits.values() for part_files in parts.values() for f in part_files]
assert len(all_split) == len(set(all_split)) == len(files["empty"]) + len(files["person"])
split_table

## Figure (a): one 2-s window per class
Here we try to visualize a single 2-second window from the empty and person sample files as side-by-side heatmaps. The color of each square is how strong the signal is at that subcarrier.

In [ ]:
centred = {}
for name in ("empty", "person"):
    w = window(first[name], start_s=10)
    centred[name] = w - np.median(w, axis=0)     # different subcarries sit at differnt amplitude levels so by subtracting median every subcarriers level is 0
lim = np.percentile(np.abs(np.vstack(list(centred.values()))), 99)   # shared colour range, ignores spikes

fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True, constrained_layout=True)
for ax, name, title in zip(axes, ("empty", "person"),
                           ("Empty room (label 0)", "One person present (label 1)")):
    c = centred[name]
    im = ax.imshow(c.T, aspect="auto", origin="lower", cmap="RdBu_r", vmin=-lim, vmax=lim,
                   extent=[0, WIN_S, 1, 48])
    ax.set_title(title)
    ax.set_xlabel("time in window (s)")
axes[0].set_ylabel("subcarrier index (0 and 48 dropped)")
fig.colorbar(im, ax=axes, label="amplitude - subcarrier median")
plt.savefig(f"{FIG_DIR}/fig_a_heatmap.png", dpi=150)
plt.show()

## Figure (b): spread per recording, empty vs person

In [ ]:
empty_spreads, person_spreads = summary["empty"][1], summary["person"][1]
fig, ax = plt.subplots(figsize=(5, 4), constrained_layout=True)
ax.boxplot([empty_spreads, person_spreads])
ax.set_xticks([1, 2], [f"Empty (0)\n{len(empty_spreads)} files", f"Person (1)\n{len(person_spreads)} files"])
ax.set_ylabel("median IQR across subcarriers")
ax.set_yscale("log")          # values span about 7 to 300
ax.set_title("Typical amplitude spread per recording, room4")
plt.savefig(f"{FIG_DIR}/fig_b_spread_boxplot.png", dpi=150)
plt.show()